In [1]:
# ============================================================
# CrimePulse_7Nation — Synthetic Dirty Data Generator + MySQL Export
# Run this directly in a Jupyter Notebook (paste as one cell, or split by the
# "# %%" markers into separate cells).
#
# Produces (star schema):
#   FACT TABLES (950,000 base rows each, ~2-3% dirty duplicates added):
#     - crime_incident_fact
#     - victim_suspect_fact
#     - case_resolution_fact
#     - financial_digital_fact
#   DIMENSION TABLES (small, clean):
#     - dim_country
#     - dim_crime_type
#     - dim_date
#
# All fact tables share crime_id as the join key.
# ============================================================

# %%
import numpy as np
import pandas as pd
import random
from datetime import date, timedelta
from sqlalchemy import create_engine, URL
import time

np.random.seed(42)
random.seed(42)

# ------------------------------------------------------------
# DB CONNECTION (PostgreSQL) — replace placeholders with YOUR
# real values. Do not commit real credentials to GitHub.
# Requires: pip install psycopg2-binary
# The database itself (e.g. crimepulse_7nation) must already
# exist — create it once via DBeaver or `CREATE DATABASE ...;`
# ------------------------------------------------------------
DB_USER     = "Abishek"
DB_PASSWORD = "@b!$#3k@2003"
PG_HOST     = "localhost"
DB_PORT     = 3306
DB_NAME     = "CrimeRecordsDB"

connection_url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=PG_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(connection_url)

N = 950_000  # base rows per fact table

# %%
# ------------------------------------------------------------
# REFERENCE DATA
# ------------------------------------------------------------
countries = ["Germany", "India", "USA", "Japan", "China", "Russia", "Canada"]

crime_types = [
    "Armed Robbery", "Forcible Abduction", "Intentional Homicide", "Manslaughter",
    "Embezzlement", "Currency Counterfeiting", "Capital Flight", "Cyber Warfare",
    "Ransomware Extortion", "Drug Trafficking", "Corporate Espionage"
]

digital_types = {"Cyber Warfare", "Ransomware Extortion", "Corporate Espionage",
                  "Currency Counterfeiting", "Capital Flight"}
financial_types = {"Embezzlement", "Currency Counterfeiting", "Capital Flight",
                    "Corporate Espionage", "Cyber Warfare", "Ransomware Extortion"}
crossborder_prone = {"Currency Counterfeiting", "Capital Flight", "Drug Trafficking",
                      "Cyber Warfare", "Corporate Espionage"}

# per-country crime type weighting (index-aligned with crime_types)
country_weights_raw = {
    "Germany": [5, 4, 4, 3, 16, 14, 6, 8, 7, 9, 18],
    "Japan":   [4, 3, 4, 3, 15, 13, 6, 7, 6, 8, 19],
    "China":   [5, 4, 5, 4, 6, 8, 15, 20, 14, 7, 16],
    "Russia":  [6, 5, 6, 5, 7, 9, 16, 19, 13, 8, 15],
    "India":   [16, 6, 5, 4, 7, 6, 5, 10, 12, 20, 4],
    "USA":     [15, 5, 4, 3, 6, 5, 4, 9, 14, 22, 5],
    "Canada":  [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9],
}
country_weights = {c: np.array(w) / sum(w) for c, w in country_weights_raw.items()}

cities = {
    "Germany": {"Berlin": "DE-BE", "Munich": "DE-BY", "Hamburg": "DE-HH",
                "Frankfurt": "DE-HE", "Cologne": "DE-NW", "Stuttgart": "DE-BW"},
    "India": {"Mumbai": "IN-MH", "Delhi": "IN-DL", "Bangalore": "IN-KA",
              "Chennai": "IN-TN", "Kolkata": "IN-WB", "Hyderabad": "IN-TG"},
    "USA": {"New York": "US-NY", "Los Angeles": "US-CA", "Chicago": "US-IL",
            "Houston": "US-TX", "Phoenix": "US-AZ", "Miami": "US-FL"},
    "Japan": {"Tokyo": "JP-13", "Osaka": "JP-27", "Yokohama": "JP-14",
              "Nagoya": "JP-23", "Sapporo": "JP-01", "Fukuoka": "JP-40"},
    "China": {"Beijing": "CN-11", "Shanghai": "CN-31", "Guangzhou": "CN-44",
              "Shenzhen": "CN-44", "Chengdu": "CN-51", "Wuhan": "CN-42"},
    "Russia": {"Moscow": "RU-MOW", "Saint Petersburg": "RU-SPE", "Novosibirsk": "RU-NVS",
               "Yekaterinburg": "RU-SVE", "Kazan": "RU-TA", "Sochi": "RU-KDA"},
    "Canada": {"Toronto": "CA-ON", "Vancouver": "CA-BC", "Montreal": "CA-QC",
               "Calgary": "CA-AB", "Ottawa": "CA-ON", "Edmonton": "CA-AB"},
}

lat_lon_ranges = {
    "USA":    ((24, 49), (-125, -66)),
    "Canada": ((43, 70), (-141, -52)),
    "Germany":((47, 55), (6, 15)),
    "India":  ((8, 37), (68, 97)),
    "Japan":  ((30, 45), (130, 145)),
    "China":  ((18, 53), (73, 135)),
    "Russia": ((41, 82), (19, 190)),
}

agencies = {
    "Germany": ["Bundeskriminalamt", "LKA Bayern", "LKA Berlin", "Zollkriminalamt", "Landespolizei Hessen"],
    "India": ["National Crime Records Bureau", "CBI", "State CID Maharashtra", "Delhi Police", "Cyber Cell India"],
    "USA": ["FBI Field Office", "DEA", "Local PD", "State Police", "US Marshals Service"],
    "Japan": ["National Police Agency", "Metropolitan Police Dept", "Prefectural Police", "Cyber Crime Unit Japan"],
    "China": ["Ministry of Public Security", "Municipal PSB", "Cyber Security Bureau", "Provincial Police"],
    "Russia": ["Ministry of Internal Affairs", "FSB Regional Unit", "Investigative Committee", "Municipal Police"],
    "Canada": ["RCMP", "Ontario Provincial Police", "Local Municipal Police", "CSIS Liaison Unit"],
}

weapons = ["Firearm", "Knife", "None", "Digital Tool", "Chemical", "Unknown"]

data_sources_clean = ["NationalCrimeDB_v2", "InterpoolFeed", "LocalPD_Report", "CrimeStat_Sync"]
data_sources_dirty = ["NationalCrimeDB_V2", "Interpol_Feed", "LocalPD_Reprot", "crimestat_sync"]

# year weighting to simulate real trend fluctuation, 2010 - 30 May 2026
year_weight_map = {
    2010: 0.8, 2011: 0.85, 2012: 0.9, 2013: 0.95,
    2014: 1.1, 2015: 1.2, 2016: 1.3, 2017: 1.4,
    2018: 1.6, 2019: 1.8, 2020: 1.7,
    2021: 1.3, 2022: 1.4, 2023: 1.5,
    2024: 1.35, 2025: 1.3, 2026: 0.55,  # partial year (Jan-May only)
}
years = list(year_weight_map.keys())
year_probs = np.array(list(year_weight_map.values()))
year_probs = year_probs / year_probs.sum()

city_typos = {"Mumbai": "Mumbay", "Tokyo": "Tokio", "Moscow": "Mosscow", "Berlin": "Berln"}

print("Reference data loaded.")

# %%
# ------------------------------------------------------------
# CORE GENERATION (vectorized where possible)
# ------------------------------------------------------------
def gen_dates(n):
    yrs = np.random.choice(years, size=n, p=year_probs)
    dates = []
    for i, y in enumerate(yrs):
        if y == 2026:
            m = np.random.randint(1, 6)  # Jan-May
        else:
            m = np.random.randint(1, 13)
        # days in month (approx, safe for all months incl. Feb)
        if m == 2:
            max_day = 29 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 28
        elif m in (4, 6, 9, 11):
            max_day = 30
        else:
            max_day = 31
        if y == 2026 and m == 5:
            max_day = min(max_day, 30)
        d = np.random.randint(1, max_day + 1)
        dates.append((y, m, d))
        if (i + 1) % 200_000 == 0:
            print(f"  ...dates generated: {i+1:,}")
    return dates

def format_dirty_date(y, m, d):
    fmt = random.choice(["dmy", "mdy", "ymd"])
    if fmt == "dmy":
        return f"{d:02d}/{m:02d}/{y}"
    elif fmt == "mdy":
        return f"{m:02d}-{d:02d}-{y}"
    else:
        return f"{y}/{m:02d}/{d:02d}"

def make_typo(word):
    if len(word) < 4:
        return word
    i = random.randint(1, len(word) - 2)
    return word[:i] + word[i+1] + word[i] + word[i+2:]

print("Starting core field generation...")
t0 = time.time()

crime_id = np.array([f"CR-{i:07d}" for i in range(1, N + 1)])
country_arr = np.random.choice(countries, size=N)

date_tuples = gen_dates(N)
incident_year = np.array([t[0] for t in date_tuples])
incident_month = np.array([t[1] for t in date_tuples])
incident_day = np.array([t[2] for t in date_tuples])
incident_date_str = np.array([format_dirty_date(*t) for t in date_tuples])

# ~1% year mismatch
mismatch_idx = np.random.choice(N, size=int(N * 0.01), replace=False)
displayed_year = incident_year.copy()
displayed_year[mismatch_idx] = displayed_year[mismatch_idx] + np.random.choice([-1, 1], size=len(mismatch_idx))

city_arr, region_arr, lat_arr, lon_arr, crime_type_arr = [], [], [], [], []

for i, c in enumerate(country_arr):
    city_choices = list(cities[c].keys())
    city = random.choice(city_choices)
    city_arr.append(city)
    region_arr.append(cities[c][city])
    lat_range, lon_range = lat_lon_ranges[c]
    lat_arr.append(round(np.random.uniform(*lat_range) + np.random.uniform(-2, 2), 5))
    lon_arr.append(round(np.random.uniform(*lon_range) + np.random.uniform(-2, 2), 5))
    crime_type_arr.append(np.random.choice(crime_types, p=country_weights[c]))
    if (i + 1) % 200_000 == 0:
        print(f"  ...city/geo/crime_type rows: {i+1:,}")

crime_type_arr = np.array(crime_type_arr)
print(f"Core fields done in {time.time()-t0:.1f}s")

# %%
# ------------------------------------------------------------
# SEVERITY, VICTIM/SUSPECT, RESOLUTION, FINANCIAL FIELDS
# ------------------------------------------------------------
severity_score = np.random.randint(1, 11, size=N)
outlier_idx = np.random.choice(N, size=int(N * 0.005), replace=False)
severity_score[outlier_idx] = np.random.choice([0, 11], size=len(outlier_idx))

def score_to_severity(s):
    if s <= 3: return "Low"
    if s <= 6: return "Medium"
    if s <= 8: return "High"
    return "Critical"

crime_severity = np.array([score_to_severity(s) for s in severity_score])

victim_count = np.random.randint(1, 51, size=N).astype(float)
suspect_count = np.random.randint(0, 21, size=N).astype(float)
case_duration_days = np.random.randint(0, 3651, size=N).astype(float)
crime_status_arr = np.random.choice(["Solved", "Unsolved", "Under Investigation", "Closed"], size=N)
arrested_flag_arr = np.random.choice(["Yes", "No"], size=N, p=[0.45, 0.55])
weapon_used_arr = np.random.choice(weapons, size=N)
reporting_agency_arr = np.array([random.choice(agencies[c]) for c in country_arr])
data_source_arr = np.random.choice(data_sources_clean + data_sources_dirty, size=N,
                                    p=[0.2, 0.2, 0.2, 0.2, 0.05, 0.05, 0.05, 0.05])

financial_loss = np.zeros(N)
is_financial = np.array([ct in financial_types for ct in crime_type_arr])
financial_loss[is_financial] = np.round(np.random.exponential(scale=50000, size=is_financial.sum()), 2)

digital_crime_flag = np.where(np.isin(crime_type_arr, list(digital_types)), "Yes", "No")
cb_base_prob = np.where(np.isin(crime_type_arr, list(crossborder_prone)), 0.35, 0.05)
cross_border_flag = np.where(np.random.rand(N) < cb_base_prob, "Yes", "No")

print("Severity / victim / resolution / financial fields done.")

# %%
# ------------------------------------------------------------
# INJECT DIRTY DATA
# ------------------------------------------------------------
def null_out(arr, frac, dtype=object):
    arr = arr.astype(dtype)
    idx = np.random.choice(len(arr), size=int(len(arr) * frac), replace=False)
    arr[idx] = None
    return arr

# country dirty variants (~3%)
country_dirty = country_arr.astype(object).copy()
dirty_map = {"USA": ["usa", "U.S.A", "U.S.A."], "India": ["INDIA", "india "],
             "China": ["china ", "CHINA"], "Germany": ["GERMANY", "germany"],
             "Japan": ["japan", "JAPAN "], "Russia": ["russia", "RUSSIA "],
             "Canada": ["canada", "CANADA "]}
idx = np.random.choice(N, size=int(N * 0.03), replace=False)
for i in idx:
    base = country_dirty[i]
    if base in dirty_map:
        country_dirty[i] = random.choice(dirty_map[base])

# crime_type dirty casing/typos (~3%)
crime_type_dirty = crime_type_arr.astype(object).copy()
idx = np.random.choice(N, size=int(N * 0.03), replace=False)
for i in idx:
    variant = random.choice(["lower", "upper", "typo"])
    v = crime_type_dirty[i]
    crime_type_dirty[i] = v.lower() if variant == "lower" else v.upper() if variant == "upper" else make_typo(v)

# crime_severity dirty variants (~3%)
severity_dirty = crime_severity.astype(object).copy()
sev_variant_map = {"Low": "low", "High": "HIGH", "Medium": "med", "Critical": "Crtical"}
idx = np.random.choice(N, size=int(N * 0.03), replace=False)
for i in idx:
    severity_dirty[i] = sev_variant_map.get(severity_dirty[i], severity_dirty[i])

# city typos (~2%)
city_dirty = np.array(city_arr, dtype=object)
idx = np.random.choice(N, size=int(N * 0.02), replace=False)
for i in idx:
    c = city_dirty[i]
    city_dirty[i] = city_typos.get(c, make_typo(c))

# arrested_flag dirty variants (~4%)
arrested_dirty = arrested_flag_arr.astype(object).copy()
arrest_variant_map = {"Yes": ["yes", "YES", "y", True], "No": ["No", "N", False, "no"]}
idx = np.random.choice(N, size=int(N * 0.04), replace=False)
for i in idx:
    arrested_dirty[i] = random.choice(arrest_variant_map[arrested_dirty[i]])

# nulls
crime_status_dirty = null_out(crime_status_arr, 0.05)
victim_count_dirty = null_out(victim_count, 0.04, dtype=float)
suspect_count_dirty = null_out(suspect_count, 0.04, dtype=float)
weapon_used_dirty = null_out(weapon_used_arr, 0.05)
case_duration_dirty = null_out(case_duration_days, 0.04, dtype=float)
financial_loss_dirty = null_out(financial_loss, 0.03, dtype=float)

# negative / extreme outliers
idx = np.random.choice(N, size=int(N * 0.01), replace=False)
victim_count_dirty[idx] = -np.abs(np.nan_to_num(victim_count_dirty[idx], nan=5.0))

idx = np.random.choice(N, size=int(N * 0.01), replace=False)
case_duration_dirty[idx] = -np.abs(np.nan_to_num(case_duration_dirty[idx], nan=10.0))

idx = np.random.choice(N, size=int(N * 0.005), replace=False)
financial_loss_dirty[idx] = -500

idx = np.random.choice(N, size=int(N * 0.002), replace=False)
financial_loss_dirty[idx] = 999_999_999

print("Dirty data injection complete.")

# %%
# ------------------------------------------------------------
# ASSEMBLE FACT TABLES
# ------------------------------------------------------------
crime_incident_fact = pd.DataFrame({
    "crime_id": crime_id,
    "incident_date": incident_date_str,
    "incident_year": displayed_year,
    "incident_month": incident_month,
    "incident_day": incident_day,
    "country": country_dirty,
    "city": city_dirty,
    "region_code": region_arr,
    "latitude": lat_arr,
    "longitude": lon_arr,
    "crime_type": crime_type_dirty,
    "crime_severity": severity_dirty,
    "severity_score": severity_score,
})

victim_suspect_fact = pd.DataFrame({
    "crime_id": crime_id,
    "victim_count": victim_count_dirty,
    "suspect_count": suspect_count_dirty,
    "arrested_flag": arrested_dirty,
    "weapon_used": weapon_used_dirty,
})

case_resolution_fact = pd.DataFrame({
    "crime_id": crime_id,
    "crime_status": crime_status_dirty,
    "case_duration_days": case_duration_dirty,
    "reporting_agency": reporting_agency_arr,
    "data_source": data_source_arr,
})

financial_digital_fact = pd.DataFrame({
    "crime_id": crime_id,
    "financial_loss_usd": financial_loss_dirty,
    "digital_crime_flag": digital_crime_flag,
    "cross_border_flag": cross_border_flag,
})

# ~2-3% duplicate rows per fact table (independent sampling, mimics real dirty joins)
def add_duplicates(df, frac=0.025):
    dup_rows = df.sample(frac=frac, replace=True, random_state=random.randint(0, 9999))
    return pd.concat([df, dup_rows], ignore_index=True)

crime_incident_fact = add_duplicates(crime_incident_fact)
victim_suspect_fact = add_duplicates(victim_suspect_fact)
case_resolution_fact = add_duplicates(case_resolution_fact)
financial_digital_fact = add_duplicates(financial_digital_fact)

print(f"crime_incident_fact: {len(crime_incident_fact):,} rows")
print(f"victim_suspect_fact: {len(victim_suspect_fact):,} rows")
print(f"case_resolution_fact: {len(case_resolution_fact):,} rows")
print(f"financial_digital_fact: {len(financial_digital_fact):,} rows")

# %%
# ------------------------------------------------------------
# DIMENSION TABLES (clean)
# ------------------------------------------------------------
dim_country = pd.DataFrame({
    "country_id": range(1, len(countries) + 1),
    "country_name": countries,
})

dim_crime_type = pd.DataFrame({
    "crime_type_id": range(1, len(crime_types) + 1),
    "crime_type_name": crime_types,
    "is_digital_crime": [ct in digital_types for ct in crime_types],
    "is_financial_crime": [ct in financial_types for ct in crime_types],
})

start_d, end_d = date(2010, 1, 1), date(2026, 5, 30)
all_days = [start_d + timedelta(days=x) for x in range((end_d - start_d).days + 1)]
dim_date = pd.DataFrame({
    "date_key": [d.strftime("%Y%m%d") for d in all_days],
    "full_date": all_days,
    "year": [d.year for d in all_days],
    "month": [d.month for d in all_days],
    "day": [d.day for d in all_days],
    "quarter": [(d.month - 1) // 3 + 1 for d in all_days],
})

print("Dimension tables built.")

# %%
# ------------------------------------------------------------
# EXPORT TO MYSQL
# (to_sql with chunksize handles the batching internally in one
#  single script call — no manual multi-file batching needed.)
# ------------------------------------------------------------
tables = {
    "crime_incident_fact": crime_incident_fact,
    "victim_suspect_fact": victim_suspect_fact,
    "case_resolution_fact": case_resolution_fact,
    "financial_digital_fact": financial_digital_fact,
    "dim_country": dim_country,
    "dim_crime_type": dim_crime_type,
    "dim_date": dim_date,
}

for name, df in tables.items():
    t0 = time.time()
    df.to_sql(name, con=engine, if_exists="replace", index=False, chunksize=20_000)
    print(f"Exported {name} ({len(df):,} rows) in {time.time()-t0:.1f}s")

print("crime_records data generated and exported to MySQL successfully.")

Reference data loaded.
Starting core field generation...
  ...dates generated: 200,000
  ...dates generated: 400,000
  ...dates generated: 600,000
  ...dates generated: 800,000
  ...city/geo/crime_type rows: 200,000
  ...city/geo/crime_type rows: 400,000
  ...city/geo/crime_type rows: 600,000
  ...city/geo/crime_type rows: 800,000
Core fields done in 236.3s
Severity / victim / resolution / financial fields done.
Dirty data injection complete.
crime_incident_fact: 973,750 rows
victim_suspect_fact: 973,750 rows
case_resolution_fact: 973,750 rows
financial_digital_fact: 973,750 rows
Dimension tables built.


OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 3306 failed: received invalid response to SSL negotiation: J

(Background on this error at: https://sqlalche.me/e/20/e3q8)